# Notebook 02 — Label Engineering

Before we can train any model, we need to turn the raw `Finding Labels` column into a clean binary target:

- **0** → No Finding (healthy)
- **1** → Disease Present (any condition)

This notebook works through that process step by step and checks for any edge cases in the data before we commit to a labeling rule.

---

**Before running**, make sure you have the credentials file at:
```
configs/nih-xray-ml-8be81f9160f0.json
```

## Step 1 — Connect to BigQuery

Same setup as Notebook 01.

In [7]:
from pathlib import Path

import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

CRED_FILENAME = "nih-xray-ml-8be81f9160f0.json"
PROJECT_ID    = "nih-xray-ml"

search_roots = [Path.cwd(), *Path.cwd().parents[:4]]
cred_path = next(
    (r / "configs" / CRED_FILENAME for r in search_roots if (r / "configs" / CRED_FILENAME).exists()),
    None
)

if cred_path is None:
    raise FileNotFoundError(
        f"Could not find '{CRED_FILENAME}' in any configs/ folder. "
        "Ask the team lead for this file and place it in configs/."
    )

credentials = service_account.Credentials.from_service_account_file(str(cred_path))
bq_client   = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print("Connected to BigQuery.")

Connected to BigQuery.


## Step 2 — Pull the full metadata

We pull all 112k rows so we can look at the complete label distribution, not just a sample.

In [8]:
query = """
SELECT
    `Image Index`,
    `Finding Labels`
FROM `nih-xray-ml.nih_xray.metadata`
"""

df = bq_client.query(query).to_dataframe(create_bqstorage_client=False)

print(f"Total rows: {len(df):,}")
df.head()

Total rows: 112,120


,Image Index,Finding Labels
0,00010360_005.png,Infiltration|Pneumonia
1,00022010_000.png,Cardiomegaly|Effusion
2,00009621_006.png,Consolidation|Infiltration
3,00016292_003.png,Consolidation|Pleural_Thickening
4,00013492_000.png,Consolidation|Edema


## Step 3 — Check for edge cases

Before assigning labels, we need to answer one important question:

> Does any image have **both** `No Finding` and a disease label at the same time?

For example, does `No Finding|Infiltration` exist? If it does, we need a rule for how to handle it.
If it does not exist, our labeling rule is simple and clean.

In [9]:
# Find any rows where the label contains BOTH "No Finding" and something else
mixed_labels = df[
    df["Finding Labels"].str.contains("No Finding") &
    (df["Finding Labels"] != "No Finding")
]

print(f"Rows with mixed 'No Finding' + disease label: {len(mixed_labels)}")

if len(mixed_labels) > 0:
    print("\nSample mixed rows:")
    print(mixed_labels["Finding Labels"].value_counts().head(10))
else:
    print("No mixed labels found — safe to apply a simple rule.")

Rows with mixed 'No Finding' + disease label: 0
No mixed labels found — safe to apply a simple rule.


## Step 4 — Create the binary label

Since no image has both `No Finding` and a disease label, the rule is simple:

- `Finding Labels == "No Finding"` → **0** (healthy)
- Anything else → **1** (disease present)

We add this as a new column called `disease_present`.

In [10]:
df["disease_present"] = (df["Finding Labels"] != "No Finding").astype(int)

print(df["disease_present"].value_counts())
print(f"\nClass balance: {df['disease_present'].mean():.1%} disease, {1 - df['disease_present'].mean():.1%} no finding")

disease_present
0    60361
1    51759
Name: count, dtype: int64

Class balance: 46.2% disease, 53.8% no finding


## Step 5 — Train / Validation / Test Split

Divide the data into three groups:

- **Train (70%)** — the model learns from this
- **Validation (15%)** — used during training to check progress without touching the test set
- **Test (15%)** — held out completely until the very end to measure final performance

**Important to Understand:** The NIH dataset has multiple X-rays per patient. We split at the **patient level** — all images from one patient go into the same group. If the same patient appeared in both train and test, the model could recognize that patient instead of learning actual disease patterns. That would make our results look better than they really are (data leakage).

The patient ID is the first part of the filename — for example `00001072_000.png` belongs to patient `00001072`.

In [12]:
#%pip install scikit-learn --quiet
from sklearn.model_selection import train_test_split

# Extract patient ID from the filename (e.g. "00001072_000.png" → "00001072")
df["patient_id"] = df["Image Index"].str.split("_").str[0]

# Split unique patients into train / val / test
patients = df["patient_id"].unique()

train_patients, temp_patients = train_test_split(patients, test_size=0.30, random_state=42)
val_patients,   test_patients = train_test_split(temp_patients, test_size=0.50, random_state=42)

# Assign each image to a split based on its patient
df["split"] = "train"
df.loc[df["patient_id"].isin(val_patients),  "split"] = "val"
df.loc[df["patient_id"].isin(test_patients), "split"] = "test"

# Show the size and class balance of each split
for split in ["train", "val", "test"]:
    subset = df[df["split"] == split]
    print(f"{split:6}: {len(subset):6,} images | {subset['disease_present'].mean():.1%} disease")

train : 77,640 images | 46.0% disease
val   : 17,083 images | 46.1% disease
test  : 17,397 images | 46.9% disease


Splits for training and testing are now complete